# Incoming Request Rate & Images-per-Request (Batch Size) Analysis

Analyzes the outputs of `scripts/incoming_request_rate_and_batch_size_analysis.py`
(run on the cloud instance, outputs copied back here). Replaces the illustrative
placeholder numbers (`R=0.1 req/s/device`, assumed `K` values) used in
`dis_group_l_buffer_ram_vs_redis_native_analysis.md` and
`dis_group_l_batch_processing_proposal.md`'s round-trip/fleet-scale cost model
with real measured values.

**A "request" here is one reconstructed arrival** (consecutive same-`dis` images,
capped at `GROUP_SIZE_MAX`), not one raw image row -- this is the same `K`
(images-per-request) variable the round-trip cost model needs a real
distribution for. See the script's own docstring for the full reconstruction
rule and its caveats (`GROUP_SIZE_MAX`'s origin, etc.).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Point this at wherever the script's OUTPUT_DIR was copied back to locally.
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "outputs")
OUT_DIR = os.getcwd()  # this notebook's own folder, for saved plots

print(f"DATA_DIR = {DATA_DIR}")

## Load the three outputs

All three are intentionally small regardless of fleet size -- bounded by device
count, time-bucket count, and max batch size respectively, not by raw image/row
count (see the script's docstring).

In [ ]:
per_device = pd.read_csv(os.path.join(DATA_DIR, "per_device_summary.csv"))
per_device["timestamp_min"] = pd.to_datetime(per_device["timestamp_min"])
per_device["timestamp_max"] = pd.to_datetime(per_device["timestamp_max"])

overall_bucket = pd.read_csv(os.path.join(DATA_DIR, "overall_request_rate_by_bucket.csv"))
overall_bucket["bucket_start"] = pd.to_datetime(overall_bucket["bucket_start"])

batch_hist = pd.read_csv(os.path.join(DATA_DIR, "images_per_request_histogram.csv"))

n_devices = len(per_device)
total_requests = int(per_device["n_requests"].sum())
total_images = int(per_device["n_images"].sum())
full_span = per_device["timestamp_max"].max() - per_device["timestamp_min"].min()

print(f"Devices: {n_devices:,}")
print(f"Total requests (arrivals): {total_requests:,}")
print(f"Total images: {total_images:,}")
print(f"Timestamp span covered: {per_device['timestamp_min'].min()} -> {per_device['timestamp_max'].max()}  ({full_span})")

## 1. Incoming request rate, per granularity (1s / 1min / 1h)

Fleet-wide (all devices combined) request count per bucket -- this is the
aggregate `N x R` the round-trip cost model's fleet-scale section needs,
measured directly rather than assumed.

In [ ]:
GRANULARITIES = ["1s", "1min", "1h"]

rate_summary_rows = []
for label in GRANULARITIES:
    s = overall_bucket.loc[overall_bucket["granularity"] == label, "n_requests"]
    desc = s.describe(percentiles=[.5, .9, .95, .99])
    rate_summary_rows.append({"granularity": label, **desc.to_dict(), "total_requests": int(s.sum())})

rate_summary = pd.DataFrame(rate_summary_rows).set_index("granularity")
rate_summary

In [ ]:
fig, axes = plt.subplots(len(GRANULARITIES), 1, figsize=(14, 3 * len(GRANULARITIES)), sharex=False)
for ax, label in zip(axes, GRANULARITIES):
    sub = overall_bucket[overall_bucket["granularity"] == label].sort_values("bucket_start")
    ax.plot(sub["bucket_start"], sub["n_requests"], linewidth=0.8)
    ax.set_title(f"Fleet-wide incoming requests per {label} bucket")
    ax.set_ylabel("requests")
axes[-1].set_xlabel("time")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "overall_request_rate.png"), dpi=120)
plt.show()

### The `R` value for the round-trip cost model

`R` = requests/sec/device, averaged across every device that was actually
active (not divided by the whole theoretical fleet size `N`, which would
understate it for devices with gaps in activity) -- this is what should
replace the illustrative `R=0.1` used previously.

In [ ]:
active_seconds = (per_device["timestamp_max"] - per_device["timestamp_min"]).dt.total_seconds()
# np.nan, not pd.NA - mixing pd.NA into a float64 Series silently upcasts it
# to object dtype on this pandas version, which makes .describe() below
# produce categorical-style (count/unique/top/freq) output instead of the
# numeric percentiles we actually want.
per_device_r = per_device["n_requests"] / active_seconds.replace(0, np.nan)

print(f"Per-device R (requests/sec), among devices with >1 request:")
print(per_device_r.describe(percentiles=[.5, .9, .95, .99]))

fleet_mean_r = per_device_r.mean()
print(f"\nMean R across devices: {fleet_mean_r:.5f} requests/sec/device")
print(f"  -> one request every {1/fleet_mean_r:.1f} seconds, per device, on average")

## 2. Images per request (batch size, `K`)

Fleet-wide distribution of how many images land in one reconstructed request.

In [ ]:
batch_hist = batch_hist.sort_values("images_per_request").reset_index(drop=True)
batch_hist["pct_of_requests"] = 100 * batch_hist["n_requests"] / batch_hist["n_requests"].sum()

weighted_mean_k = (batch_hist["images_per_request"] * batch_hist["n_requests"]).sum() / batch_hist["n_requests"].sum()
print(f"Fleet-wide mean images/request (K): {weighted_mean_k:.3f}")
print(f"Fraction of requests with K=1 (single image): {batch_hist.loc[batch_hist['images_per_request'] == 1, 'pct_of_requests'].sum():.1f}%")
print(f"Fraction of requests with K>1 (multi-image): {batch_hist.loc[batch_hist['images_per_request'] > 1, 'pct_of_requests'].sum():.1f}%")
batch_hist

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(batch_hist["images_per_request"].astype(str), batch_hist["n_requests"])
ax.set_title("Images per request (batch size, K) -- fleet-wide")
ax.set_xlabel("images per request (K)")
ax.set_ylabel("number of requests")
for i, row in batch_hist.iterrows():
    ax.text(i, row["n_requests"], f"{row['pct_of_requests']:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "images_per_request_histogram.png"), dpi=120)
plt.show()

## 3. Plugging measured R and K into the round-trip cost model

Using the corrected formulas from `dis_group_l_buffer_ram_vs_redis_native_analysis.md`:
`RAM-copy = 4 + 6P` round trips/request, `ZSET-native = 2K + 7P + 2`. `P`
(pops/request) isn't directly measured here -- in steady state (buffer already
at `embed_buffer_max_size`), `P` approaches `K` for the common case where every
arriving image eventually triggers a pop; adjust `P_ASSUMPTION` below if a
different steady-state behavior is known.

In [ ]:
N_DEVICES_ASSUMED = n_devices  # override this with the real target fleet size if different from the sample
P_ASSUMPTION = weighted_mean_k  # steady-state: P approaches K on average - see markdown above

ram_copy_round_trips = 4 + 6 * P_ASSUMPTION
zset_native_round_trips = 2 * weighted_mean_k + 7 * P_ASSUMPTION + 2

aggregate_requests_per_sec = N_DEVICES_ASSUMED * fleet_mean_r
ram_copy_cmd_per_sec = aggregate_requests_per_sec * ram_copy_round_trips
zset_native_cmd_per_sec = aggregate_requests_per_sec * zset_native_round_trips

print(f"N (devices)              = {N_DEVICES_ASSUMED:,}")
print(f"R (req/sec/device, mean) = {fleet_mean_r:.5f}")
print(f"K (images/request, mean) = {weighted_mean_k:.3f}")
print(f"P (assumed, steady-state)= {P_ASSUMPTION:.3f}")
print()
print(f"Aggregate request rate (N x R)    : {aggregate_requests_per_sec:,.1f} req/sec")
print(f"RAM-copy round trips/request      : {ram_copy_round_trips:.2f}")
print(f"ZSET-native round trips/request   : {zset_native_round_trips:.2f}")
print(f"RAM-copy aggregate commands/sec   : {ram_copy_cmd_per_sec:,.0f}")
print(f"ZSET-native aggregate commands/sec: {zset_native_cmd_per_sec:,.0f}")
print(f"With Redis+Valkey dual-write (x2) : RAM-copy = {2*ram_copy_cmd_per_sec:,.0f} cmd/s,  ZSET-native = {2*zset_native_cmd_per_sec:,.0f} cmd/s")

## Final summary

In [ ]:
print("=" * 90)
print("FINAL SUMMARY")
print("=" * 90)
print(f"Devices analyzed                 : {n_devices:,}")
print(f"Total requests (arrivals)        : {total_requests:,}")
print(f"Total images                     : {total_images:,}")
print(f"Measured mean R (req/sec/device) : {fleet_mean_r:.5f}")
print(f"Measured mean K (images/request) : {weighted_mean_k:.3f}")
print(f"K=1 share of requests            : {batch_hist.loc[batch_hist['images_per_request'] == 1, 'pct_of_requests'].sum():.1f}%")
print()
for label in GRANULARITIES:
    row = rate_summary.loc[label]
    print(f"[{label} buckets] mean={row['mean']:.2f}  median={row['50%']:.1f}  p95={row['95%']:.1f}  max={row['max']:.0f}")